In [ ]:
%load_ext autoreload
%autoreload 2

from blockymetamaterials.plotting import generate_animation, generate_frames, plot_geometry, generate_patch_collection, generate_mode_images
from blockymetamaterials.utils import SolutionData, EigenmodeData, ControlParams, GeometricalParams, LigamentParams, StretchingTorsionalSpringParams, MechanicalParams
from blockymetamaterials.analysis_utils import *
from blockymetamaterials.geometry import RotatedSquareGeometry, compute_inertia, rotation_matrix, compute_edge_angles, compute_edge_lengths

import matplotlib as mpt
import matplotlib.legend as mlegend
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from matplotlib.colors import SymLogNorm, Normalize, ListedColormap, BoundaryNorm
import matplotlib.pyplot as plt
from matplotlib import animation, cm, colors
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import matplotlib.path as mpath

import numpy as np
from numpy import sin, cos, pi, linspace, sqrt
import math
import scipy
import sympy
import sys

mpt.use('pgf')

%matplotlib widget

plt.rc('text', usetex=True)
plt.rcParams.update({'font.size': 14})
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath} \usepackage{xcolor}'


In [ ]:
# Drive frequency arrays
exp_freqs = np.arange(2, 36, 1)
exp_w = 2 * np.pi * exp_freqs

# Create mask for periodic regime: starting at timepoints = 0.064 and stopping at 0.064 + 10/freqs
# Relaxation regime starts at relax_pt = (0.064 + 10/freq)*1000 (index of timepoints array)
relax_pt = np.array([math.ceil((0.064 + 10/freq)*1000) for freq in exp_freqs])
exp_periodic_mask = []

for i, freq in enumerate(exp_freqs):
    exp_timepoints = np.load(f'out/audrey/FINAL EXP AND NONLIN DATASETS/raw_exp_data_fields/30_deg_sample/{freq}.0Hz/10periods_sine_timepoints.npy')
    end_time = 0.064 + 10/freq
    exp_periodic_mask.append((exp_timepoints >= 0.064) & (exp_timepoints <= end_time))

In [ ]:
# DEFINE EXPERIMENTAL PARAMETERS FOR RS SAMPLE

exp_config = experimental_parameters()
spacing = exp_config["spacing"]
hinge_length = exp_config["hinge_length"]

k_rot = exp_config["k_rot"]
k_stretch = exp_config["k_stretch"]
k_shear = exp_config["k_shear"]

density = exp_config["density"]

angle30 = 30 * np.pi / 180
angle5 = 5 * np.pi / 180

n1 = exp_config["n1_blocks"]//2
n2 = exp_config["n2_blocks"]//2
geometry = RotatedSquareGeometry(
    n1_cells=n1,
    n2_cells=n2, 
    spacing=spacing, 
    bond_length=hinge_length
    )
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()


B30, G1, G2, M30 = RSmoduli(angle30, spacing, k_rot, k_stretch, k_shear, hinge_length)
rho30 = exp_config["density"]/(2 * np.cos(angle30)**2)
print("30 degree sample:")
print("Coarse-grained mass density: ", f"{rho30:.2e} Mg/mm^2")
print("Effective bulk, shear 2 and shear 1 moduli: ", f"{B30:.2f}", f"{G2:.2f}", f"{G1:.2f}")
print("Effective dilation-gradient modulus: ", f"{M30:.2f}")

B5, G1, G2, M5 = RSmoduli(angle5, spacing, k_rot, k_stretch, k_shear, hinge_length)
rho5 = exp_config["density"]/(2 * np.cos(angle5)**2)
print("5 degree sample:")
print("Coarse-grained mass density: ", f"{rho5:.2e} Mg/mm^2")
print("Effective bulk, shear 2 and shear 1 moduli: ", f"{B5:.2f}", f"{G2:.2f}", f"{G1:.2f}")
print("Effective dilation-gradient modulus: ", f"{M5:.2f}")


# Mass matrix
_inertia30 = compute_inertia(
            vertices=centroid_node_vectors(angle30),
            density=density
        ).reshape((geometry.n_blocks * 3,))
print('Inertia:', _inertia30[:3])

_inertia5 = compute_inertia(
            vertices=centroid_node_vectors(angle5),
            density=density
        ).reshape((geometry.n_blocks * 3,))
print('Inertia:', _inertia5[:3])

# Fig 4(b,c,d,e)

## Fig 4b Dispersion Plot

In [ ]:
n_modes = 400
mode_range = np.arange(n_modes)+1


n1 = exp_config["n1_blocks"]//2
n2 = exp_config["n2_blocks"]//2
# Lattice dimensions
Lx = 2*n1 * spacing
Ly = 2*n2 * spacing

# Energy band (w^2) for bulk modes (using G1 and G2) of wavenumber <nx, ny>
def E2(nx, ny):
    return (np.pi**2) * G2 * (nx**2/(Lx)**2 + ny**2/(Ly)**2) / rho30
def E1(nx, ny):
    return (np.pi**2) * G1 * (nx**2/(Lx)**2 + ny**2/(Ly)**2) / rho30

# Energy eigenvalues for the first n_modes modes, sorted in ascending order
Emodes = [E1(i,j) for i in range(1,40) for j in range(1,40)]+[E2(i,j) for i in range(1,40) for j in range(1,40)]
Emodes.sort()

# Eigenfrequencies for the bulk modes in Hz
fbulk = [math.sqrt(Emodes[m])/(2*np.pi) for m in range(len(mode_range))]
print(fbulk[:5])

# Conformal ceiling - Eigenfrequency for the lowest bulk mode (nx=1, ny=1)
fceil = math.sqrt(E2(1,1))/(2*np.pi)


scale = 3

Bgood, G1, G2, Mgood = RSmoduli(angle30, spacing, k_rot/100, k_stretch, k_shear, hinge_length)
print(Bgood,G2,G1,Mgood)

def E2(nx, ny):
    return (np.pi**2) * G2 * (nx**2/(Lx*scale)**2 + ny**2/(Ly*scale)**2) / rho30
def E1(nx, ny):
    return (np.pi**2) * G1 * (nx**2/(Lx*scale)**2 + ny**2/(Ly*scale)**2) / rho30


Emodes = [E1(i,j) for i in range(1,40) for j in range(1,40)]+[E2(i,j) for i in range(1,40) for j in range(1,40)]
Emodes.sort()

fbulk_good = [math.sqrt(Emodes[m])/(2*np.pi) for m in range(400)]
print(fbulk_good[:5])
print(math.sqrt(E2(1,1))/(2*np.pi))

In [ ]:
filename_exp30 = 'notebooks/data/linear_undamped_eigenmode_analysis/eigdata_24x16_initialangle_30.0_clamped.npz'
filename_better30 = 'notebooks/data/linear_undamped_eigenmode_analysis/eigdata_krotby100_72x48_initialangle_30.0_clamped.npz'

data_exp30 = np.load(filename_exp30, allow_pickle=False)
exp_eigs = data_exp30["eigenvalues"]
exp_freq = np.sqrt(exp_eigs)/(2*np.pi)

data_better30 = np.load(filename_better30, allow_pickle=False)
good_eigs = data_better30["eigenvalues"]
good_freq = np.sqrt(good_eigs)/(2*np.pi)


# find linear fit with zero intercept for good_freq vs mode number for the first n modes
n_modes = 10

mode_arr = np.arange(0,n_modes)+1
freq_arr = good_freq[:n_modes]
slope, intercept = np.polyfit(mode_arr, freq_arr, 1, full=True)[0]
print(f"Slope: {slope}, Intercept: {intercept}")


In [ ]:
fig4b, ax4b = plt.subplots(figsize=(3.2,3.8), tight_layout=True)

analytic_good = np.loadtxt(f"notebooks/data/analytical_linear_conformal_modes/72x48_clamped12blocks/analyticfrequencies_krotby100.csv", delimiter=",", dtype=float)
analytic_real = np.loadtxt(f"notebooks/data/analytical_linear_conformal_modes/24x16_clamped12blocks/analyticfrequencies.csv", delimiter=",", dtype=float)

linear = ax4b.plot(np.arange(1,100), slope*np.arange(1,100) + intercept, 'k-', lw=1, alpha=0.5)


# IDEAL LATTICE - 3X SCALE
tpt = 10

goodline, = ax4b.plot(np.arange(tpt)+1, analytic_good[:tpt], 'bo--', label=r'theory conformal',
    markersize=3.5, mfc='white', alpha=0.5)
goodbulk, = ax4b.plot(np.arange(400)+tpt-1, fbulk_good[:400], 'ro--', label=r'theory bulk',
    markersize=3.5, mfc='white', alpha=0.5)
goodsimline, = ax4b.plot(np.arange(400)+1, good_freq[:400], 'ko--', label=r'simulation', 
    markersize=3.5, mfc='white', alpha=0.5)
good_ceil = ax4b.axhline(y=fbulk_good[0], linestyle='--', color='green', lw=1 )


# EXPERIMENTAL PARAMETERS - 1X SCALE
tpt = 4
realline, = ax4b.plot(np.arange(tpt)+1, analytic_real[:tpt], 'bo-', label=r'theory conformal',
    markersize=3.5)
realbulk, = ax4b.plot(np.arange(400)+tpt-1, fbulk[:400], 'ro-', label=r'theory bulk',
    markersize=3.5)
realsimline, = ax4b.plot(np.arange(400)+1, exp_freq[:400], 'ko-', label=r'simulation',
    markersize=3.5)

exp_ceil = ax4b.axhline(y=fbulk[0], linestyle='-', color='green', lw=1 )


ax4b.set_xlabel(r'mode number $m$')
ax4b.set_ylabel(r'mode frequency $f_m$ (Hz)')

ax4b.set_xscale('log')
ax4b.set_yscale('log')

ax4b.set_ylim(0.5, 400)
ax4b.set_xlim(0.9, 50)

ax4b.set_xticks([1,10,50])
ax4b.set_xticklabels([r'$1$',r'$10$',r'$50$'])

ax4b.set_yticks([0.5,1,5,10,30,100])
ax4b.set_yticklabels([r'$0.5$',r'$1$',r'$5$',r'$10$',r'$30$',r'$100$'])


cline, = ax4b.plot([],[], 's', c='b', markersize=10, label=r'continuum conformal')
bulk, = ax4b.plot([],[], 's', c='r', markersize=10, label=r'continuum bulk')
simline, = ax4b.plot([],[], 's', c='k', markersize=10, label=r'discrete')

dashed, = ax4b.plot([],[], 'ko--', markersize=3.5, mfc='white', alpha = 0.5, label=r'$9N$, $k_{\theta}/100$')
solid, = ax4b.plot([],[], 'ko-', markersize=3.5, label=r'$N$, $k_{\theta}$')
linfit, = ax4b.plot([],[], 'k-', markersize=3.5, alpha = 0.5, label=r'linear fit')


legend1 = ax4b.legend(handles = [cline, bulk, simline], title_fontsize=14, fontsize=14, framealpha=0,
    loc='upper left', borderpad=0, labelspacing=0.2, handletextpad=0.4, handlelength=1., borderaxespad=.3)
legend1._legend_box.align = "left"

ax4b.legend(handles = [solid, dashed, linfit], loc = 'lower right', fontsize=14, ncol=1, framealpha = 0,
    borderpad=0, labelspacing=0.2, handletextpad=0.4, handlelength=1., borderaxespad=.2,
    alignment='right', markerfirst=False)

ax4b.add_artist(legend1)

# ax4b.set_title(r'red dots w/o lattice effects')

## Fig 4c Conformal Fitting for Experimental and Better Parameters (REMOVE PICKLED LOADS)

In [ ]:
n_modes = 100
mode_range = np.arange(n_modes)+1

n1 = exp_config["n1_blocks"]//2
n2 = exp_config["n2_blocks"]//2

Lx = 2*n1 * spacing
Ly = 2*n2 * spacing
def E2(nx, ny):
    return (np.pi**2) * G2 * (nx**2/(Lx)**2 + ny**2/(Ly)**2) / rho30
def E1(nx, ny):
    return (np.pi**2) * G1 * (nx**2/(Lx)**2 + ny**2/(Ly)**2) / rho30

Emodes = [E1(i,j) for i in range(1,40) for j in range(1,40)]+[E2(i,j) for i in range(1,40) for j in range(1,40)]
Emodes.sort()

fbulk = [math.sqrt(Emodes[m])/(2*np.pi) for m in range(len(mode_range))]
print(fbulk[:5])
fceil = math.sqrt(E2(1,1))/(2*np.pi)

In [ ]:
# LOAD CONFORMAL FITTING DATA FOR BOTH CASES
folder = r'notebooks/data'
good_name = f'krotby100_72x48_initialangle_{angle30*180/np.pi:.1f}_clamped'             # Large Lattice Size and Small B
exp_name = f'24x16_initialangle_{angle30*180/np.pi:.1f}_clamped'

error_exp = np.load(f'{folder}/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample/conformal_polynomial_fiterrors_nc15.npy')
error_good = np.load(f'{folder}/conformal_fitting_analysis/linear_eigenmodes_30_deg_sample_krotby100_sizex3/conformal_polynomial_fiterrors_nc15.npy')

# LOAD STRAINS CQALCULATED BASED ON NEW SYMMETRIC UNIT-CELL DEFINITION
straindata = np.load(f'{folder}/strains_linear_eigenmodes/(grad,diln,s1,s2,stot,rotn)_{good_name}_100modes.npz')
d = straindata['d']
stot = straindata['stot']

rms_ds = np.sqrt(np.nansum(d**2, axis=tuple(range(1, d.ndim))))
rms_stot = np.sqrt(np.nansum(stot**2, axis=tuple(range(1, stot.ndim))))
rms_etot = np.sqrt(rms_ds**2 + rms_stot**2)
frac_shear_good = rms_stot/rms_etot

straindata = np.load(f'{folder}/strains_linear_eigenmodes/(grad,diln,s1,s2,stot,rotn)_{exp_name}_100modes.npz')
d = straindata['d']
stot = straindata['stot']

rms_ds = np.sqrt(np.nansum(d**2, axis=tuple(range(1, d.ndim))))
rms_stot = np.sqrt(np.nansum(stot**2, axis=tuple(range(1, stot.ndim))))
rms_etot = np.sqrt(rms_ds**2 + rms_stot**2)
frac_shear_exp = rms_stot/rms_etot

## Fig 4d Varying $\frac{B}{G_2}$ Calculations

In [ ]:
def get_conformal_fit_error(fields, geometry, initial_angle, n_modes=0, nc=15):
    n1_blocks = geometry.n1_blocks
    n2_blocks = geometry.n2_blocks
    
    masks = [remove_layers(n1_blocks, n2_blocks, i) for i in range(3)]
    
    block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()
    
    bcs = block_centroids(initial_angle)
    z_origin = np.array((bcs[0, 0] + 1j*bcs[0, 1])/2 + (bcs[n1_blocks*n2_blocks-1, 0] + 1j*bcs[n1_blocks*n2_blocks-1, 1]))/2

    if n_modes==0:
        n_modes = fields.shape[0]
    fields = fields[:n_modes].reshape((n_modes, n1_blocks*n2_blocks ,-1))

    error_arr = np.empty((3, n_modes))
    coeffs_arr = np.empty((3), dtype=object)

    for j in range(3):
        zs = np.array((bcs[masks[j], 0] + 1j*bcs[masks[j], 1] - z_origin))
        
        u_comfields = fields[:,masks[j],0] + 1j*fields[:,masks[j],1]

        coeffs_arr[j], ufits, error_arr[j] = conformal_fit(zs, Lx, u_comfields, nc, err_calc=True, frac = True)
    
    return coeffs_arr, np.sqrt(error_arr)

In [ ]:
# CALCULATE CONFORMAL FITTING ERRORS FOR MODES OF VARYING k_rot SYSTEMS

n1 = exp_config["n1_blocks"]//2
n2 = exp_config["n2_blocks"]//2

mode = 4
pow_index = np.arange(-8,6)
eigs = []
evecs = []

for i in range(len(pow_index)):
    eigs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/{2*n1}x{2*n2}_initialangle_30.0_varyingkrot_clamped/all_eigenvalues/krot_times_pow(2,{pow_index[i]}).npy')[:mode])
    evecs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/{2*n1}x{2*n2}_initialangle_30.0_varyingkrot_clamped/all_eigenvectors/krot_times_pow(2,{pow_index[i]}).npy')[:mode])

n_index = len(pow_index)

# CONFORMAL FITTING POLYNOMIAL HAS ORDER 15 (higher orders lead to smaller adjusted-R^2; can be as small as 10 without losing accuracy)
nc = 15

error_krot_arr = np.empty((3, mode, n_index))

fields = np.array(evecs)[:,:4].reshape((n_index,mode,4*n1*n2,-1))
print(fields.shape)

for i in range(mode):
    error_krot_arr[:,i] = get_conformal_fit_error(fields[:,i], geometry, angle30, nc=nc)[1]

In [ ]:
# CALCULATE CONFORMAL FITTING ERRORS FOR MODES OF VARYING INITIAL-ANGLE SYSTEMS

angle_arr = 5 * np.arange(1,9)
print(angle_arr)
eigs = []
evecs = []

for i in range(len(angle_arr)):
    eigs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/{2*n1}x{2*n2}_varyingangle_clamped/all_eigenvalues/{angle_arr[i]:.1f}.npy')[:mode])
    evecs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/{2*n1}x{2*n2}_varyingangle_clamped/all_eigenvectors/{angle_arr[i]:.1f}.npy')[:mode])

nc = 15

fields = np.array(evecs)[:,:mode].reshape((len(angle_arr),mode,4*n1*n2,-1))
print(fields.shape)
error_angle_arr = np.empty((3, mode, len(angle_arr)))

for i in range(angle_arr.shape[0]):
    error_angle_arr[:,:,i] = get_conformal_fit_error(fields[i], geometry, angle_arr*np.pi/180, nc=nc)[1]

In [ ]:
# Elastic modulii
B_krot, _, G2_krot, M = RSmoduli(angle30, spacing, k_rot * 2.0**pow_index, k_stretch, k_shear, hinge_length)
print(B_krot/G2_krot)

B_exp = RSmoduli(angle30, spacing, k_rot, k_stretch, k_shear, hinge_length)[0]
print(B_exp)

B_angle, _, G2_angle, M = RSmoduli(angle_arr * (np.pi/180) , spacing, k_rot, k_stretch, k_shear, hinge_length)
print(B_angle/G2_angle)

B_5 = RSmoduli(5*np.pi/180, spacing, k_rot, k_stretch, k_shear, hinge_length)[0]
print(B_5)

## Fig 4e Varying N Calculations

In [ ]:
def get_conformal_fit_error(fields, geometry, initial_angle, n_modes=0, nc=15):
    n1_blocks = geometry.n1_blocks
    n2_blocks = geometry.n2_blocks
    Lx = n1_blocks*spacing
    
    masks = [remove_layers(n1_blocks, n2_blocks, i) for i in range(1)]
    
    block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()
    
    bcs = block_centroids(initial_angle)
    z_origin = np.array((bcs[0, 0] + 1j*bcs[0, 1])/2 + (bcs[n1_blocks*n2_blocks-1, 0] + 1j*bcs[n1_blocks*n2_blocks-1, 1]))/2

    if n_modes==0:
        n_modes = fields.shape[0]
    fields = fields[:n_modes].reshape((n_modes, n1_blocks*n2_blocks ,-1))

    error_arr = np.empty((1, n_modes))
    coeffs_arr = np.empty((1), dtype=object)

    for j in range(1):
        zs = np.array((bcs[masks[j], 0] + 1j*bcs[masks[j], 1] - z_origin))
        
        u_comfields = fields[:,masks[j],0] + 1j*fields[:,masks[j],1]

        coeffs_arr[j], ufits, error_arr[j] = conformal_fit(zs, Lx, u_comfields, nc, err_calc=True, frac = True)
    
    return coeffs_arr, np.sqrt(error_arr)

In [ ]:
mode = 4
pow_index = np.arange(-8,6)
n_range = np.arange(1,10)

nc = 15
error_arr = np.empty((1, mode, len(n_range), len(pow_index)))

spacing = 10.
hinge_length = 0.5
initial_angle = 30*np.pi/180


for j in range(len(n_range)):
    eigs = []
    evecs = []

    x_n1 = 3*n_range[j]
    x_n2 = 2*n_range[j]
    x_geometry = RotatedSquareGeometry(n1_cells=x_n1, n2_cells=x_n2, spacing=spacing, bond_length=hinge_length)

    for i in range(len(pow_index)):
        eigs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/varying_N_data/{2*x_n1}x{2*x_n2}_30.0/varyingkrot_clamped/all_eigenvalues/krot_times_pow(2,{pow_index[i]}).npy', allow_pickle=False)[:mode])
        evecs.append(np.load(f'notebooks/data/linear_undamped_eigenmode_analysis/varying_N_data/{2*x_n1}x{2*x_n2}_30.0/varyingkrot_clamped/all_eigenvectors/krot_times_pow(2,{pow_index[i]}).npy', allow_pickle=False)[:mode])

    fields = np.array(evecs)[:,:mode].reshape((len(pow_index),mode,4*x_n1*x_n2,-1))
    print(fields.shape)

    for i in range(mode):
        error_arr[:,i,j] = get_conformal_fit_error(fields[:,i], x_geometry, initial_angle, nc=nc)[1]

## Combined Plot for Fig 4(c,d,e)

In [ ]:
n_modes = 100

paper_cols = ['#e6b800','#ff7500','#006a00','#960096']

light_blue = adjust_lightness('blue', amount=0.4)  # Blend 60% toward white
light_magenta = adjust_lightness('magenta', amount=0.4)
opaq = 0.6

fig, axes = plt.subplots(1,3,figsize=(12.5,4), tight_layout=True, gridspec_kw={'width_ratios': [3.5, 4.1, 4.5]})

ceil = axes[0].axvline(x=5, linestyle='-', lw=1, color = 'green', label = r'$f_{\mathrm{ceil}}$')
ceildash = axes[0].axvline(x=19, linestyle='--', lw=1, color = 'green', label = r'$f_{\mathrm{ceil}}$')

goodshearline, = axes[0].plot(np.arange(n_modes)+1, frac_shear_good[:n_modes], 'o--', lw=1,
    markersize=3.5, mfc='white', color=light_magenta,
    label=r'$\langle \Psi \rangle_{\mathrm{s}}$')
shearline, = axes[0].plot(np.arange(n_modes)+1, frac_shear_exp[:n_modes], 'o-', lw=1, color='magenta', markersize=3.5,
    label=r'$\langle \Psi \rangle_{\mathrm{s}}$')

goodfitline, = axes[0].plot(np.arange(n_modes)+1, error_good[:n_modes], '--', lw=1,
    marker='v', markersize=3.5, mfc='white', color=light_blue,
    label=r'$\Delta_{\mathrm{conf}}$')
fitline, = axes[0].plot(np.arange(n_modes)+1, error_exp[:n_modes], 'b-', lw=1, marker='^', markersize=3.5,
    label=r'$\Delta_{\mathrm{conf}}$')

axes[0].set_yscale('log')
axes[0].set_yticks([1,0.1,0.01,0.001])
axes[0].set_yticklabels([r'$1$',r'$0.1$',r'$0.01$',r'$0.001$'])

axes[0].set_xscale('log')
axes[0].set_xlim(0.8, 100)
axes[0].set_xticks([1,5,10,50,100])
axes[0].set_xticklabels([r'$1$',r'$5$',r'$10$',r'$50$',r'$100$'])

legend2 = axes[0].legend(title=r'$N, k_{\theta}$', handles = [fitline, shearline], fontsize=14,
    loc='upper left', bbox_to_anchor=(0.41, 0.45),
    title_fontsize=14, framealpha=0.8, borderpad=0.2, labelspacing=0.3, handletextpad=0.3, handlelength=1.,
    ncol = 2, columnspacing = 0.5)
legend2.get_frame().set_edgecolor('none')
legend2.get_frame().set_linewidth(0)
legend2._legend_box.align = "right"

legend3 = axes[0].legend(title=r'$9N,k_{\theta}/100$', handles = [goodfitline, goodshearline], fontsize=14,
    loc='upper left', bbox_to_anchor=(0.41, 0.23),
    title_fontsize=14, framealpha=0.8, borderpad=0.2, labelspacing=0.3, handletextpad=0.3, handlelength=1.,
    ncol = 2, columnspacing = 0.5)
legend3.get_frame().set_edgecolor('none')
legend3.get_frame().set_linewidth(0)
legend3._legend_box.align = "right"

axes[0].add_artist(legend2)

axes[0].set_xlabel(r'mode index $m$')



line = [None]*(mode)

axes[1].axvline(x=B_5/G2, color='red', ymax=1.1, alpha=0.5, linewidth=1.)
axes[1].axvline(x=B_exp/G2, color='blue', ymax=1.1, alpha=0.5, linewidth=1.)

for m in [1,2,3,0]:
    axes[1].plot(B_krot[:]/G2, error_krot_arr[0,m,:],'--', color='grey',
        lw=1, alpha=0.5)
    axes[1].plot(B_angle/G2, error_angle_arr[0,m,:],'-', color=paper_cols[m], lw=1, alpha=0.8)

    x, = axes[1].plot(B_krot[:]/G2, error_krot_arr[0,m,:], marker='^', markersize=6, linestyle='None',
        color=paper_cols[m], mfc='white')
    axes[1].plot(B_angle/G2, error_angle_arr[0,m,:], marker='o', linestyle='none', color=x.get_color(), markersize=5, markeredgecolor='black')
    
    line[m], = axes[1].plot([],[],'s',markersize=10, color=x.get_color(), label=f'${m+1}$')

# axes[1].plot(B_krot[:]/G2, 0.02*B_krot[:]/G2, 'k-', alpha=0.5)

axes[1].set_yscale('log')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'$B/G_2$')
axes[1].set_ylabel(r'$\Delta_{\mathrm{conf}}$', labelpad=-.5)
axes[1].set_xticks([1,B_5/G2,0.1,0.001,0.01,B_exp/G2])
axes[1].set_xticklabels([r'$1$',rf'{B_5/G2:.2f}',r'$0.1$',r'$0.001$',r'$0.01$',rf'{B_exp/G2:.2f}'])

fig.canvas.draw()

# Access x-axis tick labels
tick_labels = axes[1].get_xticklabels()
tick_labels[1].set_color("red")
tick_labels[5].set_color("blue")

axes[1].set_yticks([1,0.1,0.01])
axes[1].set_yticklabels([r'$1$',r'$0.1$',r'$0.01$'])
axes[1].set_ylim(None,1.2)


varyk, = axes[1].plot([],[],'^--', color='grey', lw=1, markersize=6, mfc='white',
    label=r'varying $k_{\mathrm{\theta}}$')
varytheta, = axes[1].plot([],[],'o-', color='grey', lw=1, markersize=5, alpha=1, mfc='grey', markeredgecolor='black',
    label=r'varying $\theta_{0}$')
title, = axes[1].plot([],[],' ',label=r'mode')

# legtop = axes[1].legend(title=r'Discrete model', handles=[], title_fontsize=14, fontsize=14, framealpha=0,
#     loc='lower left', bbox_to_anchor=(0, 1),
#     borderpad=0, labelspacing=0.2, handletextpad=0.4, handlelength=1., borderaxespad=0)
# legtop._legend_box.align = "left"

leg1 = axes[1].legend(handles=line, title='mode', title_fontsize=14, ncol=2, fontsize=14,
    framealpha=0, borderpad=0.2, labelspacing=0.2, handletextpad=0.2, columnspacing=0.2)
# axes[1].add_artist(legtop)

leg2 = mlegend.Legend(axes[1], [varyk,varytheta], [r'varying $k_{\mathrm{\theta}}$', r'varying $\theta_{0}$'], ncol=1, fontsize=14,
    framealpha=0, borderpad=0.2, labelspacing=0.2, handletextpad=0.6, handlelength=1.5)

leg1._legend_box._children.append(leg2._legend_box._children[1])
leg1._legend_box.align="center" # the default layout is 'center'



B_0 = RSmoduli(0 , spacing, k_rot, k_stretch, k_shear, hinge_length)[0]
B_45 = RSmoduli(45 * (np.pi/180) , spacing, k_rot, k_stretch, k_shear, hinge_length)[0]
width = (B_0-B_45)/G2

height = 0.00001
ax2 = axes[1].inset_axes([B_45/G2, 1.3, width, height], transform=axes[1].transData)

# Set ticks for the inner x-axis
mticks = B_angle[[0,1,3,5,7]]
ax2.set_xticks(np.log(mticks/G2))  # Custom x ticks
ax2.set_xticklabels([rf'${theta:.0f}^{{\circ}}$' for theta in angle_arr[[0,1,3,5,7]]], fontsize=13)  # Set labels and rotate if necessary
ax2.set_xticks(np.log(B_angle[[2,4,6]]/G2), minor=True)

ax2.xaxis.set_ticks_position('top')  # Position the ticks at the bottom
ax2.spines['top'].set_color('k')  # Hide the top spine
ax2.spines['right'].set_color('none')  # Hide the right spine
ax2.spines['left'].set_color('none')  # Hide the left spine
ax2.spines['bottom'].set_color('none')  # Set bottom spine color for visibility

ax2.xaxis.get_ticklabels()[0].set_color("red")
ax2.xaxis.get_ticklabels()[3].set_color("blue")

# Place the label at a custom position
ax2.text(-0.12, 20, r'$\theta_0$',              # label text
        transform=ax2.get_yaxis_transform(), ha='left', fontsize=14)

ax2.plot([],[],' ',label=r'')
ax2.legend(frameon=False)

ax2.yaxis.set_visible(False)  # Hide the y-axis
ax2.tick_params(axis='y', which='both', left=False, right=False)  # Hide tick marks

ax2.set_xlim(np.log(B_45/G2),np.log(B_0/G2))




# cmap = plt.cm.magma_r
cmap = plt.cm.viridis_r
# cmaplist = [cmap(0.1+math.sqrt(i)*0.2) for i in range(len(n_range))]
low, high = 0., 0.9
cmaplist = [cmap(low + (high-low)*i/(len(n_range)-1)) for i in range(len(n_range))]
print(len(cmaplist))

m=1
cmaplist[3] = paper_cols[m]

# Create colormap and normalization
cmap2 = ListedColormap(cmaplist)
bounds = np.linspace(0, 9, 10)  # 8 bins
norm = BoundaryNorm(bounds, cmap2.N)

size = [6,6,6,6]

axes[2].axvline(x=B_5/G2, color='red', ymax=1.1, alpha=0.5, linewidth=1.)
axes[2].axvline(x=B_exp/G2, color='blue', ymax=1.1, alpha=0.5, linewidth=1.)

for j in range(len(n_range)):
    axes[2].plot(B_krot[:]/G2, error_arr[0,m,j,:], '--', color=cmaplist[j], lw=1, alpha=0.5)

    x, = axes[2].plot(B_krot[:]/G2, error_arr[0,m,j,:], marker='^', color=cmaplist[j], markersize=6, linestyle='None', mfc='white')

mode2, = axes[2].plot([],[],linestyle='',markersize=10, color=paper_cols[m], label=f'mode ${m+1}$')

# axes[2].plot(B_krot[:]/G2, 0.02*B_krot[:]/G2, 'k-', alpha=0.5)

axes[2].set_yscale('log')
axes[2].set_xscale('log')
axes[2].set_xlabel(r'$B/G_2$')
axes[2].set_ylabel(r'$\Delta_{\mathrm{conf}}$')
axes[2].set_xticks([1,B_5/G2,0.1,0.001,0.01,B_exp/G2])
axes[2].set_xticklabels([r'$1$',rf'{B_5/G2:.2f}',r'$0.1$',r'$0.001$',r'$0.01$',rf'{B_exp/G2:.2f}'])

# fig3d.canvas.draw()

# Access x-axis tick labels
tick_labels = axes[2].get_xticklabels()

tick_labels[1].set_color("red")
tick_labels[5].set_color("blue")

axes[2].set_yticks([1,0.1,0.01])
axes[2].set_yticklabels([r'$1$',r'$0.1$',r'$0.01$'])
axes[2].set_ylim(None,1.2)

# legend1 = axes[2].legend(title=r'Discrete model', handles=[], title_fontsize=14, fontsize=14, framealpha=0,
#     loc='lower left', bbox_to_anchor=(0, 1),
#     borderpad=0, labelspacing=0.2, handletextpad=0.4, handlelength=1., borderaxespad=0)
# legend1._legend_box.align = "left"

leg1 = axes[2].legend(handles=[mode2], loc='lower right', bbox_to_anchor=(0.93, 0.3), fontsize=14,
    framealpha=0., borderpad=0., labelspacing=0.2, handletextpad=-0.5)

# axes[2].add_artist(legend1)

# leg2 = mlegend.Legend(axes[2], [varyk,varytheta], [r'varying $k_{\mathrm{rot}}$', r'varying $\theta_{0}$'], ncol=1, fontsize=14,
#     framealpha=0, borderpad=0., labelspacing=0.2, handletextpad=0.6, handlelength=1.5)

# leg1._legend_box._children.append(leg2._legend_box._children[1])
# leg1._legend_box.align="center" # the default layout is 'center'

# Add discrete colorbar
cbar = axes[2].figure.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap2), ax=axes[2],
                    ticks=np.arange(len(n_range)) + 0.5, pad=0.02)
cbar.ax.set_yticklabels([rf"${6*n_range[i]*4*n_range[i]}$" for i in range(len(n_range))])
cbar.set_label(r"$N$", rotation=0)


from matplotlib.offsetbox import OffsetImage, AnnotationBbox
image = plt.imread("notebooks/output/for_presenting/mode2_inset.png")

# Create an inset image
imbox = OffsetImage(image, zoom=.3, interpolation='antialiased')  # zoom adjusts size
ab = AnnotationBbox(imbox, (1.6, 0.013),  # (x, y) location in data coords
                    frameon=False,  # frame around the inset image
                    pad=0.3,       # padding between image and frame
                    box_alignment=(0.5, 0.5))

axes[2].add_artist(ab)


# fig.text(
#     0.12, 0.95,          # x, y in figure coordinates
#     r'Discrete Model Predictions',           # the text
#     va='center',         # vertical alignment
#     ha='center',         # horizontal alignment
#     rotation='horizontal',
#     fontsize=16,
#     fontweight='bold',
#     bbox=dict(
#         facecolor='white',  # background color
#         edgecolor='white', # optional border
#         boxstyle='round,pad=0.3',  # rounded box with padding
#     )
# )

In [ ]:
fig.savefig("notebooks/output/for_presenting/fig3_cde_v2.pdf", dpi=600, bbox_inches='tight', transparent=True)

# Fig 5(a,b,c)

## Fig 5a Conformal Fit Plot

In [ ]:
nonlin_freqs = np.arange(2, 63, 1)
lin_freqs = 2+0.2*np.arange(300)

nc = 10

linsim30_err = np.load('notebooks/data/conformal_fitting_analysis/linear_response_30_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
linsim5_err = np.load('notebooks/data/conformal_fitting_analysis/linear_response_5_deg_sample_freqrange(2,60,0.2)Hz/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')

delta_30sim = np.load('notebooks/data/conformal_fitting_analysis/nonlinear_30_deg_sample_extended-freq-range/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
delta_5sim = np.load('notebooks/data/conformal_fitting_analysis/nonlinear_5_deg_sample_extended-freq-range/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')

delta_30exp = np.load('notebooks/data/conformal_fitting_analysis/experiment_30_deg_sample/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')
delta_5exp = np.load('notebooks/data/conformal_fitting_analysis/experiment_5_deg_sample/model_deltas_conformal_polynomial_fit_nc10_periodicregime.npy')


In [ ]:
col30 = 'blue'
col5 = 'red'

fig5a, ax5a = plt.subplots(figsize=(4,3.8), tight_layout = True)

linsim, = ax5a.plot([],[], ':', color='k', label=r'linear sim')
nonlinsim, = ax5a.plot([],[], '^', fillstyle='none', lw=1, markersize = 5, color='k', label=r'nonlinear sim')
exp, = ax5a.plot([],[], 'o', markersize = 4.5, color='k', label=r'experiment')


# Initial angle = 30.0
line30, = ax5a.plot([],[], 's', markersize=10, color=col30, label=r'$\theta_0=30^{\circ}$')

ax5a.plot(lin_freqs, linsim30_err, ':', markersize = 4.5, color=col30)
ax5a.plot(nonlin_freqs, delta_30sim, '^', fillstyle='none', lw=1, markersize = 5, color = col30)
ax5a.plot(exp_freqs, delta_30exp, 'o', markersize = 4.5, color=col30)

# Initial angle = 5.0
line5, = ax5a.plot([],[], 's', markersize=10, color=col5, label=r'$\theta_0=5^{\circ}$')

ax5a.plot(lin_freqs, linsim5_err, ':', markersize = 4.5, color=col5)
ax5a.plot(nonlin_freqs, delta_5sim, '^', fillstyle='none', lw=1, markersize = 5, color = col5)
ax5a.plot(exp_freqs, delta_5exp, 'o', markersize = 4.5, color=col5)

fceil = 31.52420985648422
ceil = ax5a.axvline(x=fceil, linestyle='-', color='g', lw=1, label=r'$f_{\mathrm{ceil}}$')

ax5a.set_yscale('log')
ax5a.set_xlabel('drive frequency $f_{\mathrm{d}}$ (Hz)')
ax5a.set_ylabel(r'$\Delta_{\mathrm{conf}}$')

# ax5a.set_yticks([0.1,1])
# ax5a.set_yticklabels([r'$0.1$', r'$1$'])

ax5a.set_xticks([0,10,20,30,40,fceil,50,60])
ax5a.set_xticklabels([r'$0$',r'$10$',r'$20$','',r'$40$',r'$f_{\mathrm{ceil}}$',r'$50$',r'$60$'])
ax5a.xaxis.get_ticklabels()[5].set_color('green')


# ax5a.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
# ax5a.set_title(rf'conformal fitting of response using {nc} terms | WITH TRANSLATIONS')

legend1 = ax5a.legend(handles = [exp, nonlinsim, linsim], loc = 'upper left',
            fontsize=14, ncol=1, framealpha = 0, labelspacing=0.2, borderpad=0., handlelength=1.3, handletextpad=0.2)
ax5a.legend(handles = [line30, line5], loc = 'lower right', bbox_to_anchor=(1.05,-0.03) , fontsize=14, ncol=1, framealpha = 0, labelspacing=0.2, handletextpad=0.1)

ax5a.add_artist(legend1)

ax5a.set_ylim(0.05,1.1)
ax5a.set_xlim(None,63)

In [ ]:
fig5a.savefig("notebooks/output/for_presenting/fig_5a_v2.png", dpi=600, bbox_inches='tight', transparent=True)

## Fig 5b Resonance Plot

In [ ]:
# get eigenmodes
filename=rf'eigdata_24x16_initialangle_{angle30*180/np.pi:.1f}_clamped.npz'

normaldata = np.load(rf'notebooks/data/linear_undamped_eigenmode_analysis/{filename}')

# normaldata, lattice_params, mechanical_params = unpack(rf'notebooks/data/linear_undamped_eigenmode_analysis/{filename}')

def mode_excitation_response(fields: np.ndarray, mode: np.ndarray, timeseries=False):
    if timeseries is True:
        intensity = np.mean(np.einsum('ijk,jk->i', fields[:,:,:2], mode[:,:2])**2) / np.linalg.norm(mode[:,:2]) #scalar - avg over time
        return intensity
    else:
        intensity = np.abs(np.einsum('ijk,jk->i', fields[:,:,:2], mode[:,:2]))**2 / np.linalg.norm(mode[:,:2]) #vector along mode index
        return intensity


### Mode excitation in experiments

In [ ]:
fieldpath = f'out/audrey/FINAL EXP AND NONLIN DATASETS/raw_exp_data_fields/30_deg_sample'
forcepath = f'out/audrey/FINAL EXP AND NONLIN DATASETS/reaction_forces/raw_exp_data/30_deg_sample'

def load_exp_funcn(freq):
    tpts = np.load(f'{fieldpath}/{freq}.0Hz/10periods_sine_timepoints.npy')
    fields = np.load(f'{fieldpath}/{freq}.0Hz/fields.npy')
    forces = np.load(f'{forcepath}/{freq}Hz/reaction_forces.npy')

    tpts = (tpts[:-1] + tpts[1:])/ 2.0
    fields = (fields[:-1] + fields[1:]) / 2.0
    forces = (forces[:-1] + forces[1:])/ 2.0

    dt = 0.001
    fields[:,1] = np.gradient(fields[:,0], dt, axis=0)

    return tpts, fields, forces

In [ ]:
n_tpts = 5201

mode_intensity_exp = np.zeros((5,len(exp_freqs)))
total_intensity_exp = np.zeros((len(exp_freqs)))

for i in range(len(exp_freqs)):
    timepoints, fields, _ = load_exp_funcn(exp_freqs[i])

    disp_fields = fields[exp_periodic_mask[i][:-1],0]

    disp_no_nan = np.nan_to_num(disp_fields, nan=0.0)

    for mode in range(5):
        mode_intensity_exp[mode,i] = mode_excitation_response(disp_no_nan, normaldata['fields'][mode], timeseries=True)

    total_intensity_exp[i] = np.mean(np.einsum('ijk,ijk->i', disp_no_nan[:,:,:2], disp_no_nan[:,:,:2]))


### StackPlot 4 modes 

In [ ]:
import matplotlib.patches as patches
fig5b, ax5b = plt.subplots(figsize=(4.3,3.8), tight_layout = False)

# colors = ['#ffd700','#ff7500','#006a00','#960096']
# paper_cols = ['#e6b800','#ff7500','#006a00','#960096','#d3d3d3']
paper_cols = ['#e6b800','#ff7500','#006a00','#960096','grey']

othermodes = 1 - np.sum(mode_intensity_exp[:4], axis=0)/total_intensity_exp
layers = np.vstack([mode_intensity_exp[:4]/total_intensity_exp, othermodes])

# Stackplot on the Axes object
ax5b.stackplot(exp_freqs, layers, labels=[r'$1$', r'$2$', r'$3$', r'$4$', r'others'],
    colors = paper_cols)

stack_labels = [r'$1$', r'$2$', r'$3$', r'$4$', r'$\geq 5$']
stack_handles = [Patch(facecolor=c, label=l) for c, l in zip(paper_cols, stack_labels)]


# Draw contour lines along boundaries
cumulative = np.zeros_like(exp_freqs, dtype=float)
for mode in range(4):
    cumulative += layers[mode]
    ax5b.plot(exp_freqs, cumulative, '.-', color='white', linewidth=1, markersize=3.5)  # boundary line

vline, = ax5b.plot([],[],linestyle='-', lw=1, color='k', label=r'$f_m$')

mline, = ax5b.plot([],[],linestyle='none', color='k', label=r'mode $m$')

ax5b.set_ylim(0.,1e0)
ax5b.set_xlim(2,35)

ax5b.set_xticks([2,5,10,15,20,25,30,35,40])
ax5b.set_xticklabels([r'$2$',r'$5$',r'$10$','',r'$20$','',r'$30$',r'$35$',r'$40$'])

# Draw custom tick labels, shifting the one at index 2 right by 0.1 in data units
for mode in range(4):
    ax5b.axvline(x=np.sqrt(normaldata['eigenvalues'][mode])/(2*np.pi), linestyle='-', color='k', lw=2, alpha=0.6, zorder=4)

    label = rf'$\boldsymbol{{f_{mode+1}}}$'  # or any custom string
    x = np.sqrt(normaldata['eigenvalues'][mode])/(2*np.pi)
    
    ax5b.axvline(x, ymin=ax5b.get_ylim()[0]-0.013, ymax=ax5b.get_ylim()[0], color=paper_cols[mode], linewidth=2, clip_on=False, zorder=5)

    # Shift label at index 2 by +0.1
    if mode == 0:
        x = x - 0.5
    elif mode == 1:
        x = x + 0.5

    ax5b.text(x, -0.03, label, ha='center', va='top', color=paper_cols[mode], transform=ax5b.get_xaxis_transform())

line, = ax5b.plot([],[],'k-',lw=2,alpha=0.6,label=r'$f_m$')

ax5b.set_xlabel(r'drive frequency $f_{\mathrm{d}}$ (Hz)')
ax5b.set_ylabel(r'modal intensity fraction ($I_m/I_{\mathrm{exp}}$)')
header = ax5b.text(2,1.03,r'$30^{\circ}$ experiment response',color='blue')

# Let matplotlib auto-scale the y-limits from the data
ymin, ymax = ax5b.get_ylim()

# Add rectangle to highlight data region
data_box = patches.Rectangle(
    (2, ymin),                  # (x, y) = bottom-left
    33,                         # width
    ymax - ymin,               # height
    linewidth=2,
    edgecolor='blue',
    facecolor='none',
    linestyle='-',
    clip_on=False,              # <--- This avoids the clipping issue
    zorder=6
)
ax5b.add_patch(data_box)

# Remove spines
ax5b.spines['top'].set_visible(False)
ax5b.spines['right'].set_visible(False)
ax5b.spines['left'].set_visible(False)
# ax5b.spines['bottom'].set_visible(False)

# Shrink the width of the ax5b to leave space for the legend
box = ax5b.get_position()
ax5b.set_position([box.x0+box.width*0.05, box.y0+box.height*0.05, box.width*0.98, box.height*0.95])  # 80% width

# Add legends
legend1 = ax5b.legend(handles = stack_handles+[line], title=r'mode $m$', loc='lower left', bbox_to_anchor=(0.84,0.2),
    ncol=1, fontsize=14, framealpha=0, borderpad=0.4, labelspacing=0.2, handletextpad=0.4, handlelength=0.8, columnspacing=0.4)
legend1.get_title().set_horizontalalignment('left')


ax5b.add_artist(legend1)

In [ ]:
fig5b.canvas.draw()

fig5b.savefig(
    "notebooks/output/for_presenting/fig5b_v2.png",
    dpi=600,
    bbox_inches='tight',
    bbox_extra_artists=(legend1,header),
    pad_inches=0.08,
    transparent=True,
)

In [ ]:
print("Best case at", exp_freqs[np.argmin(othermodes)], 'Hz with first 4 modes covering',1-np.min(othermodes), 'of the response')

## Fig 5c Experimental Snapshot

In [ ]:
# load experimental data for snapshots
timepoints, fields, _ = load_exp_funcn(14)
disp_fields = fields[:,0]

defn_norm = np.linalg.norm(disp_fields, axis=(1,2))
print(disp_fields.shape)
tp = np.argmax(defn_norm)

print(tp/1000, 's, ', defn_norm[tp])

In [ ]:
filename = r'out/audrey/FINAL EXP AND NONLIN DATASETS/raw_exp_data_fields/experimentalsnapshots'

image_data = EigenmodeData(
        block_centroids=bcs,
        centroid_node_vectors=centroid_node_vectors(30*np.pi/180),
        eigenvalues=np.array([tp*0.001]),
        fields=disp_fields[tp][np.newaxis, :, :],
    )

xlim, ylim = geometry.get_xy_limits(30*np.pi/180) + 1. * geometry.spacing * jnp.array([-1, 1])
scale = 1.0

# cmap rebalancing function
# from matplotlib.colors import ListedColormap
# def gamma_adjusted_cmap(base_cmap_name='plasma', gamma=0.5, n_colors=256):
#     base = cm.get_cmap(base_cmap_name, n_colors)
#     # Remap input range through gamma
#     indices = np.linspace(0, 1, n_colors) ** gamma
#     new_colors = base(indices)
#     return ListedColormap(new_colors, name=f'{base_cmap_name}_gamma{gamma}')

# my_cmap = gamma_adjusted_cmap(gamma=0.7)


singlemap = ListedColormap(np.repeat([[0.5, 0.5, 0.5]], 256, axis=0))

generate_mode_images(
    data=image_data,
    field="u",
    scale_deformation=scale,
    deformed=True,
    mode_range=jnp.arange(1),
    out_dir=f"{filename}/14Hz_{tp*0.001:.3f}sec",
    figsize=(10, 7),
    xlim=xlim,
    ylim=ylim,
    geometry=geometry,
    mesh=False,
    lattice=True,
    # latt_alpha=0.75,
    cmap=singlemap,
    var='t'
)

# Fig 6(a,b,c,d)

In [ ]:
labels = [r'$z$', r'$\bar{z}$', r'$i\bar{z}$', r'$z^2$', r'$iz^2$', r'$z\bar{z}$', r'$iz\bar{z}$', r'$\bar{z}^2$', r'$i\bar{z}^2$', r'$iz$', r'$1$', r'$i$']
print(len(labels))

### Load Experiment Analysis

In [ ]:
file_exp30 = r'notebooks/data/conservation_analysis/raw_experiment_30_deg_sample'

map_norms = np.load(f'{file_exp30}/map_norms.npy')
map_norms = map_norms/np.sqrt(_inertia30[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_exp30}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_exp30}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_exp30}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m,relax_pt[f]:])/ np.sqrt(devs[f][m,relax_pt[f]:].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m,relax_pt[f]:])/ np.sqrt(map_momentum[f][m,relax_pt[f]:].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
exp30_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz


mean_exp30_rvalues = np.nanmean(exp30_rvalues, axis=1)
print(mean_exp30_rvalues.shape)
std_exp30_rvalues = np.nanstd(exp30_rvalues, axis=1)
max_exp30_rvalues = np.nanmax(exp30_rvalues, axis=1)
min_exp30_rvalues = np.nanmin(exp30_rvalues, axis=1)


# FOR FULL TIME SERIES
exp30_momrate = [np.real_if_close(map_momentumrate[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]
exp30_forces = [np.real_if_close(map_extforces[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]

In [ ]:
file_exp5 = r'notebooks/data/conservation_analysis/raw_experiment_5_deg_sample'

map_norms = np.load(f'{file_exp5}/map_norms.npy')
map_norms = map_norms/np.sqrt(_inertia5[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_exp5}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_exp5}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_exp5}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m,relax_pt[f]:])/ np.sqrt(devs[f][m,relax_pt[f]:].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m,relax_pt[f]:])/ np.sqrt(map_momentum[f][m,relax_pt[f]:].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
exp5_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz

mean_exp5_rvalues = np.nanmean(exp5_rvalues, axis=1)
std_exp5_rvalues = np.nanstd(exp5_rvalues, axis=1)
max_exp5_rvalues = np.nanmax(exp5_rvalues, axis=1)
min_exp5_rvalues = np.nanmin(exp5_rvalues, axis=1)


# FOR FULL TIME SERIES
exp5_momrate = [np.real_if_close(map_momentumrate[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]
exp5_forces = [np.real_if_close(map_extforces[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]

### Load Nonlinear Analysis

In [ ]:
file_nonlin30 = r'notebooks/data/conservation_analysis/nonlinear_30_deg_sample'

map_norms = np.load(f'{file_nonlin30}/map_norms.npy')

map_norms = map_norms/np.sqrt(_inertia30[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_nonlin30}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_nonlin30}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_nonlin30}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m,relax_pt[f]:])/ np.sqrt(devs[f][m,relax_pt[f]:].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m,relax_pt[f]:])/ np.sqrt(map_momentum[f][m,relax_pt[f]:].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
nonlin30_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz

# FOR FULL TIME SERIES
# nonlin30_momrate = [np.real_if_close(map_momentumrate[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))] 
# nonlin30_forces = [np.real_if_close(map_extforces[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))] 

In [ ]:
file_nonlin5 = r'notebooks/data/conservation_analysis/nonlinear_5_deg_sample'

map_norms = np.load(f'{file_nonlin5}/map_norms.npy')
map_norms = map_norms/np.sqrt(_inertia5[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_nonlin5}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_nonlin5}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_nonlin5}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m,relax_pt[f]:])/ np.sqrt(devs[f][m,relax_pt[f]:].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m,relax_pt[f]:])/ np.sqrt(map_momentum[f][m,relax_pt[f]:].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))

nonlin5_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz

# FOR FULL TIME SERIES
# nonlin5_momrate = [np.real_if_close(map_momentumrate[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]
# nonlin5_forces = [np.real_if_close(map_extforces[f])/map_norms[:, np.newaxis] for f in range(len(exp_freqs))]

### Load Linear Analysis

In [ ]:
file_lin30 = r'notebooks/data/conservation_analysis/linear_30_deg_sample'

map_norms = np.load(f'{file_lin30}/map_norms.npy')

map_norms = map_norms/np.sqrt(_inertia30[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_lin30}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_lin30}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_lin30}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m])/ np.sqrt(devs[f][m].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m])/ np.sqrt(map_momentum[f][m].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
lin30_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz

In [ ]:
file_lin5 = r'notebooks/data/conservation_analysis/linear_5_deg_sample'

map_norms = np.load(f'{file_lin5}/map_norms.npy')

map_norms = map_norms/np.sqrt(_inertia5[0])

map_momentum = []
map_momentumrate = []
map_extforces = []
devs = []

for i, freq in enumerate(exp_freqs):
    map_momentum.append(np.load(f'{file_lin5}/map_momenta/{freq}Hz.npy'))
    map_momentumrate.append(np.load(f'{file_lin5}/map_momentum_rates/{freq}Hz.npy'))
    map_extforces.append(np.load(f'{file_lin5}/map_ext_forces/{freq}Hz.npy'))
    
    # deviations
    devs.append(map_momentumrate[-1] - map_extforces[-1])

# FOR RELAXATION REGIME
rms_devs = np.real_if_close(np.array([[np.linalg.norm(devs[f][m])/ np.sqrt(devs[f][m].size) 
                                    for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[f][m])/ np.sqrt(map_momentum[f][m].size)
                                        for f in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
lin5_rvalues = (rms_devs/rms_momentum)/(2*np.pi)           # to find inverse-timescale in Hz

## Fig 6a Parameteric plot for Experiment Relaxing Regime

In [ ]:
i = -5
start = 0
end = 600
print(exp_freqs[i])

rel30_momrate = exp30_momrate[i][:,relax_pt[i]:]
rel5_momrate = exp5_momrate[i][:,relax_pt[i]:]

rel30_force = exp30_forces[i][:,relax_pt[i]:]
rel5_force = exp5_forces[i][:,relax_pt[i]:]

fig6a, ax6a = plt.subplots(2, figsize=(3.6, 3.2), constrained_layout = True, sharex=True)

fig6a.set_constrained_layout_pads(hspace=0.0, h_pad=0.0)

lim = np.max([np.nanmax(np.abs(rel30_force[:3])), np.nanmax(np.abs(rel30_momrate[:3])), np.nanmax(np.abs(rel5_force[:3])), np.nanmax(np.abs(rel5_momrate[:3]))])*1.2
xarr = np.arange(-lim, lim, 2*lim/100)
linear, = ax6a[0].plot(xarr, xarr, color='k', lw=1, linestyle='-', label=r'$y=x$')

ax6a[0].plot(rel5_force[2,start:end], rel5_momrate[2,start:end], '.-', markersize=3, lw=1, color='red', label=r'$i\bar{z}$')
ax6a[0].plot(rel5_force[1,start:end], rel5_momrate[1,start:end], '.-', markersize=3, lw=1, color='green', label=r'$\bar{z}$')
ax6a[0].plot(rel5_force[0,start:end], rel5_momrate[0,start:end], '.-', markersize=3, lw=1, color='blue', label=r'$z$')


ax6a[0].set_title(f'relaxing data for $f_{{\mathrm{{d}}}} = {exp_freqs[i]}$ Hz', fontsize=14)
ax6a[0].set_aspect('equal', adjustable='box')
ax6a[0].set_xlim(-1.5,1.5)
ax6a[0].set_ylim(-1,1)



# make square patches (not lines) for legend using the same colors
h1, = ax6a[0].plot([], [], 's', markersize=10, color='blue', label=r'$z$')
h2, = ax6a[0].plot([], [], 's', markersize=10, color='green', label=r'$\bar{z}$')
h3, = ax6a[0].plot([], [], 's', markersize=10, color='red', label=r'$i\bar{z}$')

# legend = ax6a[0].legend(handles=[h1, h2, h3], loc='lower right', title=r'map $f$', bbox_to_anchor=(1,0),
#     framealpha=0, borderaxespad=0.1, borderpad=0.2, handletextpad=0.4, labelspacing=0.2, handlelength=1.0)

legend = ax6a[0].legend(handles=[h1, h2, h3], loc='center left', title=r'map $f$', bbox_to_anchor=(1.01,-0.1),
    framealpha=0, borderaxespad=0.1, borderpad=0.2, handletextpad=0.4, labelspacing=0.2, handlelength=1.0)


leg30 = ax6a[0].legend(handles=[], loc='upper left', title=r'$5^{\circ}$ exp', framealpha=0, borderaxespad=0.2,  # padding between legend and axes
    borderpad=0.3, fontsize=15)
# leg30.get_title().set_color('blue')

ax6a[0].add_artist(legend)

linear, = ax6a[1].plot(xarr, xarr, color='k', lw=1, linestyle='-', label=r'$y=x$')
    
ax6a[1].plot(rel30_force[2,start:end], rel30_momrate[2,start:end], '.-', markersize=3, lw=1, color='red', label=r'$i\bar{z}$')
ax6a[1].plot(rel30_force[1,start:end], rel30_momrate[1,start:end], '.-', markersize=3, lw=1, color='green', label=r'$\bar{z}$')
ax6a[1].plot(rel30_force[0,start:end], rel30_momrate[0,start:end], '.-', markersize=3, lw=1, color='blue', label=r'$z$')

ax6a[1].set_xticks(np.arange(-2, 3, 1))
ax6a[1].set_ylim(-1,1)
ax6a[1].set_aspect('equal', adjustable='box')

ax6a[1].set_xlabel(r'force $\vec{F}_{\mathrm{ext}} \cdot \vec{f}$ ($\mathrm{N}$)')

leg30 = ax6a[1].legend(handles=[], loc='upper left', title=r'$30^{\circ}$ exp', framealpha=0, borderaxespad=0.2,  # padding between legend and axes
    borderpad=0.3, fontsize=15)

# leg5.get_title().set_color('red')

# Common ylabel
# plt.subplots_adjust(left=0.2, bottom=0.2)
ylabel_artist = fig6a.text(0.0, 0.55, r'momentum rate $\dot{Q}_f$ ($\mathrm{N}$)', va='center', rotation='vertical')



In [ ]:
# Ensure off-axes legend and shared ylabel are included in the exported bounding box.
fig6a.canvas.draw()
fig6a.savefig(
    f"notebooks/output/for_presenting/deviation_from_conservation/fig6a_{exp_freqs[i]}Hz_relaxing_v3.pdf",
    dpi=600,
    bbox_inches='tight',
    bbox_extra_artists=(legend, ylabel_artist),
    pad_inches=0.08,
    transparent=False,
)

## Fig 6b

In [ ]:
fig6b, ax6b = plt.subplots(1,2,figsize=(7.2,3.2), constrained_layout=True, sharey=True, gridspec_kw={'wspace': 0.03})

# cols = ['b', 'green', 'orange', 'r']
cols = ['b', 'green', 'r']

line = [None]*3

op = 0.6

# Plot lin. response
line[0], = ax6b[0].plot([],[], 'k-', label=r'linear')

ax6b[0].plot(exp_freqs, lin5_rvalues[0], ':', color=cols[0], label=labels[0])
ax6b[0].plot(exp_freqs, lin5_rvalues[1], ':', color=cols[1], label=labels[1])
ax6b[0].plot(exp_freqs, lin5_rvalues[2], ':', color=cols[2], label=labels[2])


line[1], = ax6b[0].plot([],[],'k^', label=r'non-linear')

ax6b[0].plot(exp_freqs, nonlin5_rvalues[0], '^', mfc='none', markersize=6, color=cols[0], label=labels[0], zorder=10)
ax6b[0].plot(exp_freqs, nonlin5_rvalues[1], '^', alpha=op, markersize=6, color=cols[1], label=labels[1])
ax6b[0].plot(exp_freqs, nonlin5_rvalues[2], '^', alpha=op, markersize=6, color=cols[2], label=labels[2])


# Plot experimental response
line[2], = ax6b[0].plot([],[], 'ko', alpha=0.5, label=r'experiment')

ax6b[0].plot(exp_freqs, exp5_rvalues[0], 'o', mfc='none', markersize=5, color=cols[0], label=labels[0], zorder=11)
ax6b[0].plot(exp_freqs, exp5_rvalues[1], 'o', markersize=5, color=cols[1], label=labels[1])
ax6b[0].plot(exp_freqs, exp5_rvalues[2], 'o', markersize=5, color=cols[2], label=labels[2])

ax6b[0].set_ylabel(r'$\tau_{\mathrm{dev}}^{-1} = \displaystyle \frac{\|\dot{Q}_f - \vec{F}_\mathrm{ext}\cdot \vec{f} \|}{\|Q_f\|}$ ($\mathrm{s}^{-1}$)')
ax6b[0].set_yscale('log')
ax6b[0].text(25, 1.5*1e4, r'$5^{\circ}$ design', color='black', fontsize=15, backgroundcolor='white')
ax6b[0].grid(True)

ax6b[0].set_xlabel(r'relaxing data indexed by $f_{\mathrm{d}} (\mathrm{Hz})$')
ax6b[0].set_xticks([2,10,20,30,35])

# Plot lin. response
line[0], = ax6b[1].plot([],[], 'k:', label=r'linear')

ax6b[1].plot(exp_freqs, lin30_rvalues[0], ':', color=cols[0], label=labels[0])
ax6b[1].plot(exp_freqs, lin30_rvalues[1], ':', color=cols[1], label=labels[1])
ax6b[1].plot(exp_freqs, lin30_rvalues[2], ':', color=cols[2], label=labels[2])


line[1], = ax6b[1].plot([],[], 'k^', mfc=(0,0,0,1), mec='k', mew=1., label=r'non-linear')

ax6b[1].plot(exp_freqs, nonlin30_rvalues[0], '^', mfc='none', markersize=6, color=cols[0], label=labels[0], zorder=10)
ax6b[1].plot(exp_freqs, nonlin30_rvalues[1], '^', alpha=op, markersize=6, color=cols[1], label=labels[1])
ax6b[1].plot(exp_freqs, nonlin30_rvalues[2], '^', alpha=op, markersize=6, color=cols[2], label=labels[2])


# Plot experimental response
line[2], = ax6b[1].plot([],[], 'ko', mfc=(0,0,0,1), mec='k', mew=1., label=r'experiment')

ax6b[1].plot(exp_freqs, exp30_rvalues[0], 'o', mfc='none', markersize=5, color=cols[0], label=labels[0], zorder=11)
ax6b[1].plot(exp_freqs, exp30_rvalues[1], 'o', markersize=5, color=cols[1], label=labels[1])
ax6b[1].plot(exp_freqs, exp30_rvalues[2], 'o', markersize=5, color=cols[2], label=labels[2])

ax6b[1].set_yscale('log')
ax6b[1].text(2, 1.5*1e4, r'$30^{\circ}$ design', color='black', fontsize=15, backgroundcolor='white')

ax6b[1].grid(True)

ax6b[1].set_xlabel(r'relaxing data indexed by $f_{\mathrm{d}} (\mathrm{Hz})$')
ax6b[1].set_xticks([2,10,20,30,35])

ax6b[1].legend(handles=line, loc='upper right', bbox_to_anchor=(1.03, 1.02), ncol=1, fontsize=14,
               labelspacing=0.3, columnspacing=0.8, handlelength=1, handletextpad=0.3, borderpad=0.,
               edgecolor='white')


In [ ]:
fig6b.savefig(f"notebooks/output/for_presenting/deviation_from_conservation/fig6b_v7_rawexp_rawnonlinear.pdf", dpi=600, bbox_inches='tight', transparent=False)

## Fig 6c Varying B/G2 plot

In [ ]:
i_range = np.arange(-8, 6)

k_rot_arr = exp_config['k_rot'] * 2.**(i_range)
n_rot_arr = exp_config['n_rot'] * 2.**(i_range)

# Vectorize the moduli function to apply it over array of k_rot
moduli_gen = np.vectorize(RSmoduli, excluded={0, 'initial_angle', 1, 'spacing', 3, 'k_stretch', 4, 'k_shear', 5, 'l'})
B_arr, G1_arr, G2_arr = moduli_gen(angle30, exp_config['spacing'], k_rot_arr, exp_config['k_stretch'], exp_config['k_shear'], exp_config['hinge_length'])[:3]
ratio_2 = B_arr/G2_arr
ratio_1 = B_arr/G1_arr

### Load Linear Varying $k_{\theta}$ Data

In [ ]:
file_lin = r'notebooks/data/conservation_analysis/linear_varyingkrot'

lin_rvalues = np.zeros((len(i_range), 12, len(exp_freqs)))
lin_momrate = np.zeros((len(i_range), 12, len(exp_freqs)))
lin_forces = np.zeros((len(i_range), 12, len(exp_freqs)))

for idx, pow in enumerate(i_range):
    map_norms = np.load(f'{file_lin}/map_norms/krotx2power{pow}.npy')

    map_momentum = []
    map_momentumrate = []
    map_extforces = []
    devs = []
    
    for j, freq in enumerate(exp_freqs):
        map_momentum.append(np.load(f'{file_lin}/map_momenta/krotx2power{pow}/{freq}Hz.npy'))
        map_momentumrate.append(np.load(f'{file_lin}/map_momentum_rates/krotx2power{pow}/{freq}Hz.npy'))
        map_extforces.append(np.load(f'{file_lin}/map_ext_forces/krotx2power{pow}/{freq}Hz.npy'))
        
        devs.append(map_momentumrate[-1] - map_extforces[-1])

    rms_devs = np.real_if_close(np.array(
        [[np.linalg.norm(devs[j][m])/ np.sqrt(devs[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))

    rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[j][m])/ np.sqrt(map_momentum[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
    lin_rvalues[idx] = (rms_devs/rms_momentum)/(2*np.pi)

    lin_momrate[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_momentumrate[j][m])/ np.sqrt(map_momentumrate[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]
    lin_forces[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_extforces[j][m])/ np.sqrt(map_extforces[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]

    # rms_momentumrate = np.real_if_close(np.array([np.sqrt(np.mean(map_momentumrate[:,f]*np.conjugate(map_momentumrate[:,f]), axis=1)) for f in range(len(exp_freqs))]).T)
    # lin_rvalues[id] = rms_devs/rms_momentumrate
    

mean_lin_rvalues = np.nanmean(lin_rvalues, axis=2)
std_lin_rvalues = np.nanstd(lin_rvalues, axis=2)
max_lin_rvalues = np.nanmax(lin_rvalues, axis=2)
min_lin_rvalues = np.nanmin(lin_rvalues, axis=2)

### Load Nonlinear Varying $k_{\theta}$ Data

In [ ]:
file_nonlin = 'notebooks/data/conservation_analysis/nonlinear_varying_krot'

amp = '4'

nonlin_rvalues = np.zeros((len(i_range), 12, len(exp_freqs)))
nonlin_momrate = np.zeros((len(i_range), 12, len(exp_freqs)))
nonlin_forces = np.zeros((len(i_range), 12, len(exp_freqs)))

for idx, pow in enumerate(i_range):
    map_norms = np.load(f'{file_nonlin}/map_norms/krotx2power{pow}.npy')

    map_momentum = []
    map_momentumrate = []
    map_extforces = []
    devs = []
    
    for j, freq in enumerate(exp_freqs):
        map_momentum.append(np.load(f'{file_nonlin}/map_momenta/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,relax_pt[j]:])
        map_momentumrate.append(np.load(f'{file_nonlin}/map_momentum_rates/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,relax_pt[j]:])
        map_extforces.append(np.load(f'{file_nonlin}/map_ext_forces/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,relax_pt[j]:])
        
        devs.append(map_momentumrate[-1] - map_extforces[-1])

    rms_devs = np.real_if_close(np.array(
        [[np.linalg.norm(devs[j][m])/ np.sqrt(devs[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))

    rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[j][m])/ np.sqrt(map_momentum[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]))
    nonlin_rvalues[idx] = (rms_devs/rms_momentum)/(2*np.pi)

    nonlin_momrate[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_momentumrate[j][m])/ np.sqrt(map_momentumrate[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]
    nonlin_forces[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_extforces[j][m])/ np.sqrt(map_extforces[j][m].size) for j in range(len(exp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]

    # rms_momentumrate = np.real_if_close(np.array([np.sqrt(np.mean(map_momentumrate[:,f]*np.conjugate(map_momentumrate[:,f]), axis=1)) for f in range(len(exp_freqs))]).T)
    # lin_rvalues[id] = rms_devs/rms_momentumrate
    

mean_nonlin_rvalues = np.nanmean(nonlin_rvalues, axis=2)
std_nonlin_rvalues = np.nanstd(nonlin_rvalues, axis=2)
max_nonlin_rvalues = np.nanmax(nonlin_rvalues, axis=2)
min_nonlin_rvalues = np.nanmin(nonlin_rvalues, axis=2)

In [ ]:
# file_nonlin = 'notebooks/data/conservation_analysis/nonlinear_varying_krot'

amp = '1e-08'

lowamp_index = np.arange(3, 34, 5)
lowamp_freqs = exp_freqs[lowamp_index]
lowamp_relax_pt = relax_pt[lowamp_index]
print(lowamp_freqs)

lowamp_nonlin_rvalues = np.zeros((len(i_range), 12, len(lowamp_freqs)))
lowamp_nonlin_momrate = np.zeros((len(i_range), 12, len(lowamp_freqs)))
lowamp_nonlin_forces = np.zeros((len(i_range), 12, len(lowamp_freqs)))

for idx, pow in enumerate(i_range):
    map_norms = np.load(f'{file_nonlin}/map_norms/krotx2power{pow}.npy')

    map_momentum = []
    map_momentumrate = []
    map_extforces = []
    devs = []
    
    for j, freq in enumerate(lowamp_freqs):
        map_momentum.append(np.load(f'{file_nonlin}/map_momenta/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,lowamp_relax_pt[j]:])
        map_momentumrate.append(np.load(f'{file_nonlin}/map_momentum_rates/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,lowamp_relax_pt[j]:])
        map_extforces.append(np.load(f'{file_nonlin}/map_ext_forces/krotx2power{pow}_amplitude_{amp}mm/{freq}Hz.npy')[:,lowamp_relax_pt[j]:])
        
        devs.append(map_momentumrate[-1] - map_extforces[-1])

    rms_devs = np.real_if_close(np.array(
        [[np.linalg.norm(devs[j][m])/ np.sqrt(devs[j][m].size) for j in range(len(lowamp_freqs))] for m in range(map_norms.shape[0])]))

    rms_momentum = np.real_if_close(np.array([[np.linalg.norm(map_momentum[j][m])/ np.sqrt(map_momentum[j][m].size) for j in range(len(lowamp_freqs))] for m in range(map_norms.shape[0])]))
    lowamp_nonlin_rvalues[idx] = (rms_devs/rms_momentum)/(2*np.pi)

    lowamp_nonlin_momrate[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_momentumrate[j][m])/ np.sqrt(map_momentumrate[j][m].size) for j in range(len(lowamp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]
    lowamp_nonlin_forces[idx] = np.real_if_close(np.array(
        [[np.linalg.norm(map_extforces[j][m])/ np.sqrt(map_extforces[j][m].size) for j in range(len(lowamp_freqs))] for m in range(map_norms.shape[0])]
        ))/map_norms[:, np.newaxis]

    # rms_momentumrate = np.real_if_close(np.array([np.sqrt(np.mean(map_momentumrate[:,f]*np.conjugate(map_momentumrate[:,f]), axis=1)) for f in range(len(lowamp_freqs))]).T)
    # lin_rvalues[id] = rms_devs/rms_momentumrate
    

mean_lowamp_nonlin_rvalues = np.nanmean(lowamp_nonlin_rvalues, axis=2)
std_lowamp_nonlin_rvalues = np.nanstd(lowamp_nonlin_rvalues, axis=2)
max_lowamp_nonlin_rvalues = np.nanmax(lowamp_nonlin_rvalues, axis=2)
min_lowamp_nonlin_rvalues = np.nanmin(lowamp_nonlin_rvalues, axis=2)

### Only Linear and Experiment Plot

In [ ]:
x_range = ratio_2
x_30 = B30/G2_arr[0]
x_5 = B5/G2_arr[0]

fig6c, ax6c = plt.subplots(figsize=(4.2,3.8),tight_layout=True, sharey=True, sharex=True)

cols = ['b', 'green', 'red']

ax6c.axvline(x=x_5, linestyle='--', color='k', ymax=1.1, alpha=0.5, linewidth=1.)
ax6c.text(x_5*.5, 0.04, r'$5^{\circ}$', color='k', fontsize=15, verticalalignment='bottom', horizontalalignment='left')
ax6c.axvline(x=x_30, linestyle='--', color='k', ymax=1.1, alpha=0.5, linewidth=1.)
ax6c.text(x_30*1.2, 0.04, r'$30^{\circ}$', color='k', fontsize=15, verticalalignment='bottom', horizontalalignment='left')


line = [None]*2

ax6c.fill_between(x_range, min_lin_rvalues[:,0], max_lin_rvalues[:,0], alpha=0.3, color=cols[0])
ax6c.fill_between(x_range, min_lin_rvalues[:,1], max_lin_rvalues[:,1], alpha=0.3, color=cols[1])
ax6c.fill_between(x_range, min_lin_rvalues[:,2], max_lin_rvalues[:,2], alpha=0.3, color=cols[2])

line[0], = ax6c.plot([],[], '.-', color='k', label=r'linear')

ax6c.plot(x_range, mean_lin_rvalues[:,0], '.-', color=cols[0])
ax6c.plot(x_range, mean_lin_rvalues[:,1], '.-', color=cols[1])
ax6c.plot(x_range, mean_lin_rvalues[:,2], '.-', color=cols[2])

ax6c.plot(x_range, 70*x_range, 'k-', lw=1, label=r'$B/G_2$')

# EXPERIMENTAL DATA POINTS
x = [x_30, x_30, x_30, x_5, x_5, x_5]
y = [mean_exp30_rvalues[0], mean_exp30_rvalues[1], mean_exp30_rvalues[2], mean_exp5_rvalues[0], mean_exp5_rvalues[1], mean_exp5_rvalues[2]]
ymin = [min_exp30_rvalues[0], min_exp30_rvalues[1], min_exp30_rvalues[2], min_exp5_rvalues[0], min_exp5_rvalues[1], min_exp5_rvalues[2]]
ymax = [max_exp30_rvalues[0], max_exp30_rvalues[1], max_exp30_rvalues[2], max_exp5_rvalues[0], max_exp5_rvalues[1], max_exp5_rvalues[2]]

yerr_low = [y[i]-ymin[i] for i in range(len(y))]
yerr_high = [ymax[i]-y[i] for i in range(len(y))]

cs = [cols[0], cols[1], cols[2], cols[0], cols[1], cols[2]]

line[1] = ax6c.errorbar([0], [0], yerr=[[0.1], [0.1]], marker='o', color='k', mfc='white', capsize=6, linestyle='none', elinewidth=1.5, label=r'exp.')
for idx in range(len(cs) - 1, -1, -1):
    ax6c.errorbar(x[idx], y[idx], yerr=[[yerr_low[idx]], [yerr_high[idx]]], marker='o', color=cs[idx], zorder=5, label=None, mfc='white', capsize=6, linestyle='none', elinewidth=1.5)

ax6c.set_ylabel(r'$\tau_{\mathrm{dev}}^{-1} = \displaystyle \frac{\|\dot{Q}_f - \vec{F}_\mathrm{ext}\cdot \vec{f} \|}{\|Q_f\|}$ ($\mathrm{s}^{-1}$)')
ax6c.set_yscale('log')
# ax6c.set_ylim(1e-3, 50)

# ax6c.grid(True)
ax6c.set_xscale('log')
ax6c.set_xlabel(r'$B/G_2$')
ax6c.set_xticks([1,x_5,0.1,0.001,0.01,x_30])
ax6c.set_xticklabels([r'$1$',rf'{x_5:.2f}',r'$0.1$',r'$0.001$',r'$0.01$',rf'{x_30:.2f}'])
ax6c.set_ylim(0.03, 2.2*1e4)

# ax6c.grid(True)

# Access x-axis tick labels
tick_labels = ax6c.get_xticklabels()

tick_labels[1].set_color("red")
tick_labels[5].set_color("blue")

leg1 = ax6c.legend(handles = line, labels=[r'linear', r'exp.'], loc='upper left', bbox_to_anchor=(0.01, 0.99), ncol=1, fontsize=14, markerfirst=False,
               labelspacing=0.4, columnspacing=0.8, handlelength=1, handletextpad=0.3, borderpad=0.2, framealpha=0)



In [ ]:
fig6c.savefig(f"notebooks/output/for_presenting/deviation_from_conservation/fig6c_v3.pdf", dpi=600, bbox_inches='tight', transparent=False)

### Linear, Nonlinear and Experiment Plot

In [ ]:
x_range = ratio_2
x_30 = B30/G2_arr[0]
x_5 = B5/G2_arr[0]

map_index = [0,1,2]
cols = ['b', 'green', 'red']
cs = cols + cols
print(cs)


fig6c, ax6c = plt.subplots(figsize=(4.2,3.8),tight_layout=True, sharey=True, sharex=True)

patch_lin = Patch(
    facecolor='gray',
    alpha=0.3,
    hatch='\\\\\\',
    edgecolor='k',
    linewidth=1,
    label=''
)

patch = Patch(
    facecolor='gray',
    alpha=0.3,
    label=''
)


ax6c.axvline(x=x_5, linestyle='--', color='k', ymax=1.1, alpha=0.2, linewidth=1.)
ax6c.text(x_5*.5, 0.04, r'$5^{\circ}$', color='k', fontsize=15, verticalalignment='bottom', horizontalalignment='left')
ax6c.axvline(x=x_30, linestyle='--', color='k', ymax=1.1, alpha=0.2, linewidth=1.)
ax6c.text(x_30*1.2, 0.04, r'$30^{\circ}$', color='k', fontsize=15, verticalalignment='bottom', horizontalalignment='left')

line = [None]*4


# LINEAR PLOT LINES AND VARIANCE BANDS
line[0], = ax6c.plot([],[], ':', lw=1, color='k', label=r'linear sim')

ax6c.fill_between(x_range, min_lin_rvalues[:,2], max_lin_rvalues[:,2], alpha=0.2, color=cols[2], edgecolor='none')
ax6c.fill_between(x_range, min_lin_rvalues[:,2], max_lin_rvalues[:,2], facecolor='none', hatch='\\\\', edgecolor=cols[2], linewidth=0.0, alpha=0.5)
ax6c.fill_between(x_range, min_lin_rvalues[:,1], max_lin_rvalues[:,1], alpha=0.2, color=cols[1], edgecolor='none')
ax6c.fill_between(x_range, min_lin_rvalues[:,1], max_lin_rvalues[:,1], facecolor='none', hatch='\\\\', edgecolor=cols[1], linewidth=0.0, alpha=0.5)
ax6c.fill_between(x_range, min_lin_rvalues[:,0], max_lin_rvalues[:,0], alpha=0.2, color=cols[0], edgecolor='none')
ax6c.fill_between(x_range, min_lin_rvalues[:,0], max_lin_rvalues[:,0], facecolor='none', hatch='\\\\', edgecolor=cols[0], linewidth=0.0, alpha=0.5)

ax6c.plot(x_range, mean_lin_rvalues[:,2], ':', lw=1, alpha=0.5, color=cols[2], zorder=3)
ax6c.plot(x_range, mean_lin_rvalues[:,1], ':', lw=1, alpha=0.5, color=cols[1], zorder=3)
ax6c.plot(x_range, mean_lin_rvalues[:,0], ':', lw=1, color=cols[0], zorder=5)


# NONLINEAR LOW-AMP PLOT LINES AND VARIANCE BANDS
line[1], = ax6c.plot([],[], 's-', markersize=5, mfc='white', lw=0.5, color='k', label=r'low-amp nonlin sim')

ax6c.fill_between(x_range, min_lowamp_nonlin_rvalues[:,2], max_lowamp_nonlin_rvalues[:,2], alpha=0.2, color=cols[2], edgecolor='k')
ax6c.fill_between(x_range, min_lowamp_nonlin_rvalues[:,1], max_lowamp_nonlin_rvalues[:,1], alpha=0.2, color=cols[1], edgecolor='k')
ax6c.fill_between(x_range, min_lowamp_nonlin_rvalues[:,0], max_lowamp_nonlin_rvalues[:,0], alpha=0.2, color=cols[0], edgecolor='k')

ax6c.plot(x_range, mean_lowamp_nonlin_rvalues[:,2], 's-', markersize=5, mfc='white', lw=0.5, color=cols[2], zorder=3)
ax6c.plot(x_range, mean_lowamp_nonlin_rvalues[:,1], 's-', markersize=5, mfc='white', lw=0.5, color=cols[1], zorder=3)
ax6c.plot(x_range, mean_lowamp_nonlin_rvalues[:,0], 's-', markersize=5, mfc='white', lw=0.5, color=cols[0], zorder=5)


# NONLINEAR PLOT LINES AND VARIANCE BANDS
line[2], = ax6c.plot([],[], '^-', markersize=5, mfc='white', lw=0.5, color='k', label=r'nonlin sim')

ax6c.fill_between(x_range, min_nonlin_rvalues[:,2], max_nonlin_rvalues[:,2], alpha=0.2, color=cols[2])
ax6c.fill_between(x_range, min_nonlin_rvalues[:,1], max_nonlin_rvalues[:,1], alpha=0.2, color=cols[1])
ax6c.fill_between(x_range, min_nonlin_rvalues[:,0], max_nonlin_rvalues[:,0], alpha=0.2, color=cols[0])

ax6c.plot(x_range, mean_nonlin_rvalues[:,2], '^-', markersize=5, mfc='white', lw=0.5, color=cols[2], zorder=3)
ax6c.plot(x_range, mean_nonlin_rvalues[:,1], '^-', markersize=5, mfc='white', lw=0.5, color=cols[1], zorder=3)
ax6c.plot(x_range, mean_nonlin_rvalues[:,0], '^-', markersize=5, mfc='white', lw=0.5, color=cols[0], zorder=5)

# ax6c.plot(x_range, 70*x_range, 'k-', lw=0.5, label=r'$B/G_2$')


# EXPERIMENTAL DATA POINTS
x = [x_30]*len(map_index) + [x_5]*len(map_index)

y = [mean_exp30_rvalues[i] for i in map_index] + [mean_exp5_rvalues[i] for i in map_index]
ymin = [min_exp30_rvalues[i] for i in map_index] + [min_exp5_rvalues[i] for i in map_index]
ymax = [max_exp30_rvalues[i] for i in map_index] + [max_exp5_rvalues[i] for i in map_index]

yerr_low = [y[i]-ymin[i] for i in range(len(y))]
yerr_high = [ymax[i]-y[i] for i in range(len(y))]

line[3] = ax6c.errorbar([0], [0], yerr=[[0.1], [0.1]], marker='o', color='k', mfc='white', capsize=6, linestyle='none', elinewidth=1.5, label=r'experiment')
for idx in range(len(cs)):
    ax6c.errorbar(x[-idx-1], y[-idx-1], yerr=[[yerr_low[-idx-1]], [yerr_high[-idx-1]]], marker='o', color=cs[-idx-1], zorder=5, label=None, mfc='white', capsize=6, linestyle='none', elinewidth=1.5)


ax6c.set_ylabel(r'$\tau_{\mathrm{dev}}^{-1} = \displaystyle \frac{\|\dot{Q}_f - \vec{F}_\mathrm{ext}\cdot \vec{f} \|}{\|Q_f\|}$ ($\mathrm{s}^{-1}$)')
ax6c.set_yscale('log')
# ax6c.set_ylim(1e-3, 50)

# ax6c.grid(True)
ax6c.set_xscale('log')
ax6c.set_xlabel(r'$B/G_2$')
ax6c.set_xticks([1,x_5,0.1,0.001,0.01,x_30])
ax6c.set_xticklabels([r'$1$',rf'{x_5:.2f}',r'$0.1$',r'$0.001$',r'$0.01$',rf'{x_30:.2f}'])
ax6c.set_ylim(0.03, 2.2*1e4)

# ax6c.grid(True)

# Access x-axis tick labels
tick_labels = ax6c.get_xticklabels()

tick_labels[1].set_color("red")
tick_labels[5].set_color("blue")

leg1 = ax6c.legend(handles = [line[1], line[2], line[0], line[3]], loc='upper left', bbox_to_anchor=(0.0, 1.02), ncol=1, fontsize=14, markerfirst=True,
               labelspacing=0.4, columnspacing=0.8, handlelength=1, handletextpad=0.3, borderpad=0.2, framealpha=0)

ax6c.legend(handles = [patch_lin, patch], loc='upper right',bbox_to_anchor=(0.9, 1.02), ncol=1, fontsize=14, markerfirst=True,
               labelspacing=0.4, columnspacing=0.8, handlelength=1, handleheight=1, handletextpad=0.3, borderpad=0.2, framealpha=0)

ax6c.add_artist(leg1)


In [ ]:
fig6c.savefig(f"notebooks/output/for_presenting/deviation_from_conservation/fig6c_v6_withlowamp.pdf", dpi=600, bbox_inches='tight', transparent=False)

## Fig 6d (NEEDS UPDATE)

In [ ]:
file_path = Path(r'notebooks/data/conservation_analysis/linear_varying_N_krotnrotby1000')

labels = [r'$z$', r'$\bar{z}$', r'$i\bar{z}$', r'$z^2$', r'$iz^2$', r'$z\bar{z}$', r'$iz\bar{z}$', r'$\bar{z}^2$', r'$i\bar{z}^2$', r'$iz$', r'$1$', r'$i$']


# SHAPE: (6 sizes, 10 samples, 12 maps, 2000 timepoints)
map_momentum = np.load(file_path / 'map_momenta.npy')
map_momentumrate = np.load(file_path / 'map_momentum_rates.npy')
map_extforces = np.load(file_path / 'map_ext_forces.npy')
map_norms = np.load(file_path / 'map_norms.npy')


print(map_momentum.shape)


devs = map_momentumrate - map_extforces
rms_devs = np.sqrt(np.mean(devs*np.conjugate(devs), axis=3))
rms_momentum = np.sqrt(np.mean(map_momentum*np.conjugate(map_momentum), axis=3))
rms_momentumrate = np.sqrt(np.mean(map_momentumrate*np.conjugate(map_momentumrate), axis=3))

lin_rvalues = np.abs(rms_devs/rms_momentum).T/(2*np.pi)
lin_momentum = np.abs(rms_momentum/map_norms[:, None,:]).T

mean_lin_rvalues = np.nanmean(lin_rvalues, axis=1)
min_lin_rvalues = np.nanmin(lin_rvalues, axis=1)
max_lin_rvalues = np.nanmax(lin_rvalues, axis=1)

print(mean_lin_rvalues.shape)

In [ ]:
n_range = np.arange(1, 7)

x_range = n_range**2 * 6*4
x_exp = 24*16

fig6d, ax6d = plt.subplots(figsize=(4.8,3.8),tight_layout=True)

# cols = ['b', 'green', 'orange', 'r']
cols = ['b', 'green', 'orange', 'r']
# comp_cols = ['#1f95b4', '#ffee00', '#ff9f0e', '#d64728']
comp_cols = ['#4a90e2', '#1b7837', '#e6550d', '#fb6a4a']

# ord2_cols = ['blue', '#00FFFF', 'green', '#00FF00', 'red', '#fb6a4a']

# ord2_cols = ['blue', 'blue', '#8B00FF', '#8B00FF', 'red', 'red']
ord2_cols = ['blue', 'blue', 'magenta', 'magenta', 'brown', 'brown']

ax6d.axvline(x=x_exp, linestyle='--', color='k', ymax=1.1, alpha=0.5, linewidth=1.)

ax6d.plot(x_range, mean_lin_rvalues[0], '.-', color=cols[0], label=labels[0])

ax6d.plot(x_range, mean_lin_rvalues[3], 'o--', color=ord2_cols[0], label=labels[3])
ax6d.plot(x_range, mean_lin_rvalues[4], 'o--', color=ord2_cols[0], markerfacecolor='white', label=labels[4])


ax6d.plot(x_range, mean_lin_rvalues[5], 'o--', color=ord2_cols[2], label=labels[5])
ax6d.plot(x_range, mean_lin_rvalues[6], 'o--', color=ord2_cols[3], markerfacecolor='white', label=labels[6])

ax6d.plot(x_range, mean_lin_rvalues[7], 'o--', color=ord2_cols[4], label=labels[7])
ax6d.plot(x_range, mean_lin_rvalues[8], 'o--', color=ord2_cols[5], markerfacecolor='white', label=labels[8])
ax6d.plot(x_range, mean_lin_rvalues[2], '.-', color=cols[3], label=labels[2])

ax6d.plot(x_range, mean_lin_rvalues[1], '.-', color=cols[1], label=labels[1])



ax6d.set_ylabel(r'$\tau_{\mathrm{dev}}^{-1} = \displaystyle \frac{\|\dot{Q}_f - \vec{F}_\mathrm{ext}\cdot \vec{f} \|}{\|Q_f\|}$ ($\mathrm{s}^{-1}$)')
ax6d.set_yscale('log')
ax6d.set_ylim(1e-2, None)

ax6d.set_xlabel(r'Lattice size $N$')

ax6d.text(600, 1e3, r'$30^\circ$ linear'+'\n'+r'$k_{\theta}/1000$', color='k', fontsize=15)
ax6d.text(390, 1e-1, r'$N_{\mathrm{exp}}$', color='k', fontsize=14)

ax6d.legend(loc='center left', bbox_to_anchor=(0.98, 0.5), title=r'map $f$', ncol=1, fontsize=14, framealpha=0,
               labelspacing=0.3, columnspacing=0.8, handlelength=1.5, handletextpad=0.3)


In [ ]:
fig6d.savefig(f"notebooks/output/for_presenting/deviation_from_conservation/fig6d_v4.pdf", dpi=600, bbox_inches='tight', transparent=False)